In [0]:
import os, sys
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), "src"))
from lh import common
from pyspark.sql import functions as F

dbutils.widgets.dropdown("env", "dev", ["dev", "test", "prod"])
env = dbutils.widgets.get("env")
cfg = common.load_config(env)
cat = cfg["catalog"]
results = []

def check(name, ok, detail=""):
    results.append(bool(ok))
    print(("PASS " if ok else "FAIL ") + name + (f" -> {detail}" if detail and not ok else ""))

g = lambda t: spark.table(f"{cat}.gold.{t}")
f = g("fact_fqf_event")
print(cat, "fact_fqf_event")

In [0]:
f.groupBy("event_type", "event_group").agg(F.count("*").alias("rows"), F.sum("amount").alias("amount")).orderBy("event_type").show()

n = f.count()
check("event_key filled and unique", f.filter("event_key IS NULL").count() == 0 and f.select("event_key").distinct().count() == n)

missing = f.join(g("dim_date"), "date_key", "left_anti").count()
check("every date_key is in dim_date", missing == 0, missing)

for key, dim in [("loan_key", "dim_loan"), ("customer_key", "dim_customer"), ("product_key", "dim_product"), ("branch_key", "dim_branch")]:
    bad = f.filter(F.col(key).isNull()).count() + f.join(g(dim).select(key), key, "left_anti").count()
    check(f"every {key} is filled and in {dim}", bad == 0, bad)

groups = {r[0]: r[1] for r in f.select("event_type", "event_group").distinct().collect()}
check("event types and groups as designed",
      groups == {"disbursement": "lending", "emi_due": "due", "repayment": "collection", "late_fee": "fee"}, groups)

check("every amount is above zero", f.filter("amount IS NULL OR amount <= 0").count() == 0)

future = f.filter((F.col("event_type") == "emi_due") & (F.to_date(F.col("date_key").cast("string"), "yyyyMMdd") > F.current_date())).count()
check("no instalment due in the future", future == 0, future)

disb = (f.filter("event_type = 'disbursement'").groupBy("loan_key").agg(F.sum("amount").alias("paid_out"))
         .join(g("dim_loan").select("loan_key", "principal"), "loan_key"))
bad = disb.filter(F.abs(F.col("paid_out") - F.col("principal")) > 1).count()
check("disbursed amount = loan principal for every loan", bad == 0, bad)
check("every loan was disbursed", disb.count() == g("dim_loan").filter("loan_key <> -1").count())

rep = (f.filter("event_type = 'repayment' AND loan_key <> -1").groupBy("loan_key").agg(F.sum("principal_part").alias("repaid"))
        .join(g("dim_loan").select("loan_key", "principal"), "loan_key"))
bad = rep.filter(F.col("repaid") > F.col("principal") + 1).count()
check("repaid principal never above loan principal", bad == 0, bad)

split = f.filter("event_type IN ('emi_due', 'repayment') AND principal_part IS NOT NULL")
bad = split.filter(F.abs(F.col("principal_part") + F.col("interest_part") - F.col("amount")) > 0.01).count()
check("principal + interest = amount for each instalment", bad == 0, bad)

orphan = f.filter("loan_key = -1")
bad = orphan.filter("customer_key <> -1 OR product_key <> -1 OR branch_key <> -1").count()
check("an unknown loan has unknown customer, product and branch", bad == 0, bad)
print("INFO events on unknown loans:", orphan.count())

bad = [c for c in ["full_name", "phone", "aadhaar", "dob", "monthly_income", "bank_ref"] if c in f.columns]
check("no personal columns", not bad, bad)
print("INFO hospital-bill disbursements:", f.filter("event_type = 'disbursement' AND is_hospital_bill").count())

print(f"\nt19_gold_fact_fqf: {sum(results)} of {len(results)} checks passed ->", "PASS" if all(results) else "FAIL")
if not all(results):
    raise RuntimeError("t19_gold_fact_fqf failed")